# Q4 FEA solver: no-terminal quickstart

Run each grey box in order with the ▶ button (or Shift+Enter).
Edit the numbers in the **SETTINGS** box and run it again to try your own beam.

## 1. Download the solver (run once)

In [ ]:
import os
if not os.path.exists("CFD"):
    !git clone -b claude/sharp-carson-1l4xvs https://github.com/nothikemu/CFD.git
%cd CFD
!pip -q install -r requirements.txt

## 2. Check it works (validation tests, ~5 s)

In [ ]:
!python -m pytest test_validation.py -q

## 3. Your beam: edit SETTINGS, then run

In [ ]:
# ----------------------------- SETTINGS (edit me) -----------------------------
LENGTH = 1.0            # m
HEIGHT = 0.1            # m
THICKNESS = 0.02        # m
E = 210e9               # Pa  (steel 210e9, aluminium 70e9)
NU = 0.3                # Poisson's ratio
YIELD = 250e6           # Pa
FORCE = -2000.0         # N   (negative = down)
NX, NY = 100, 10        # mesh: elements along length / through height
SCALE = None            # deformation exaggeration in the picture (None = automatic)
# ------------------------------------------------------------------------------

from engine import Mesh, Material, FEAModel
from visualizer import plot_results
import matplotlib.pyplot as plt

mesh = Mesh.structured_rectangle(LENGTH, HEIGHT, NX, NY)
model = FEAModel(mesh, Material(E, NU, thickness=THICKNESS, yield_strength=YIELD))
model.fix(mesh.nodes_at(x=0.0), "xy")
tip = mesh.closest_node(LENGTH, HEIGHT / 2)
model.add_nodal_force(tip, fy=FORCE)
r = model.solve()

I = THICKNESS * HEIGHT**3 / 12
print(f"tip drop (FEA)       : {r.displacements[tip, 1]:+.6e} m")
print(f"tip drop (hand calc) : {-abs(FORCE) * LENGTH**3 / (3 * E * I):+.6e} m")
print(f"max Von Mises        : {r.max_von_mises / 1e6:.2f} MPa")
print(f"safety factor        : {r.safety_factor.min():.2f}")
fig, used = plot_results(model, r, scale=SCALE, filename="output/my_beam.png")
plt.show()

## 4. Built-in benchmarks with the full step-by-step maths log

In [ ]:
!python app.py

In [ ]:
from IPython.display import Image, display
for f in ["output/cantilever_von_mises.png", "output/cantilever_components.png", "output/plate_hole_von_mises.png"]:
    display(Image(f))

## 5. More examples (see the `examples/` folder)

In [ ]:
!python examples/02_simply_supported_pressure.py
from IPython.display import Image, display
display(Image("output/02_simply_supported.png"))